One-vs-rest (OvR) turns multi-class labels into one binary label array per class (that class = test, all others = reference). No samples are dropped, so every array runs through a single ``CPP`` instance:

In [ ]:
import numpy as np
import pandas as pd
import aaanalysis as aa

labels = [0, 0, 1, 1, 2, 2]
dict_labels = aa.SequenceFeature.get_labels_ovr(labels)
dict_labels

Loop the per-class arrays through ``CPP.run`` and concatenate, tagging each ``df_feat`` with its class:

In [ ]:
df_seq = aa.load_dataset(name="DOM_GSEC", n=9)
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
multiclass = np.array([i % 3 for i in range(len(df_parts))])

cpp = aa.CPP(df_parts=df_parts)
frames = []
for cls, vec in aa.SequenceFeature.get_labels_ovr(multiclass).items():
    df_feat = cpp.run(labels=vec, n_filter=3)
    df_feat["class"] = cls
    frames.append(df_feat)
pd.concat(frames, ignore_index=True)[["class", "feature", "abs_auc"]].head()

**What can go wrong?** Labels must be integers with more than one class, and ``label_test`` must differ from ``label_ref``:

In [ ]:
for bad in [lambda: aa.SequenceFeature.get_labels_ovr([1, 1, 1]),
            lambda: aa.SequenceFeature.get_labels_ovr([0, 1], label_test=1, label_ref=1),
            lambda: aa.SequenceFeature.get_labels_ovr([0.0, 1.0, 2.0])]:
    try:
        bad()
    except ValueError as e:
        print("ValueError:", e)